### STEP 0.1 Load raw data

In [1]:
import pandas as pd
import os

# -------------------------------
# 1) ตั้งค่าให้ Pandas แสดงผลอ่านง่าย
# -------------------------------
pd.set_option('display.max_columns', None)   # แสดงทุกคอลัมน์
pd.set_option('display.width', 200)          # ความกว้างตาราง
pd.set_option('display.max_colwidth', 20)    # ความกว้างข้อความต่อคอลัมน์
pd.set_option('display.float_format', '{:.4f}'.format)  # ทศนิยมสวย ๆ

# -------------------------------
# 2) path ไปยังไฟล์ CSV
# -------------------------------
file_path = r"C:\punpun\Big data\US-Traffic-Accident-BigData-Analysis\data\raw\US_Accidents_March23.csv"

# -------------------------------
# 3) โหลดข้อมูล
# -------------------------------
if os.path.exists(file_path):
    print("พบไฟล์แล้ว กำลังโหลดข้อมูล...\n")
    
    df = pd.read_csv(file_path)
    
    print(f"โหลดสำเร็จ! จำนวนข้อมูล: {len(df):,} แถว")
    print(f"จำนวนคอลัมน์: {df.shape[1]}\n")
    
    # แสดง 5 แถวแรก แบบเห็นครบทุกคอลัมน์
    display(df.head())
else:
    print(f"ไม่พบไฟล์ที่: {os.path.abspath(file_path)}")


พบไฟล์แล้ว กำลังโหลดข้อมูล...

โหลดสำเร็จ! จำนวนข้อมูล: 7,728,394 แถว
จำนวนคอลัมน์: 46



,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.8651,-84.0587,NaN,NaN,0.0100,Right lane block...,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08 05:58:00,36.9000,NaN,91.0000,29.6800,10.0000,Calm,NaN,0.0200,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Night
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.9281,-82.8312,NaN,NaN,0.0100,Accident on Bric...,Brice Rd,Reynoldsburg,Franklin,OH,43068-3402,US,US/Eastern,KCMH,2016-02-08 05:51:00,37.9000,NaN,100.0000,29.6500,10.0000,Calm,NaN,0.0000,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Day
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.0631,-84.0326,NaN,NaN,0.0100,Accident on OH-3...,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08 06:56:00,36.0000,33.3000,100.0000,29.6700,10.0000,SW,3.5000,NaN,Overcast,False,False,False,False,False,False,False,False,False,False,False,True,False,Night,Night,Day,Day
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.7478,-84.2056,NaN,NaN,0.0100,Accident on I-75...,I-75 S,Dayton,Montgomery,OH,45417,US,US/Eastern,KDAY,2016-02-08 07:38:00,35.1000,31.0000,96.0000,29.6400,9.0000,SW,4.6000,NaN,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Day,Day,Day
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.6278,-84.1884,NaN,NaN,0.0100,Accident on McEw...,Miamisburg Cente...,Dayton,Montgomery,OH,45459,US,US/Eastern,KMGY,2016-02-08 07:53:00,36.0000,33.3000,89.0000,29.6500,6.0000,SW,3.5000,NaN,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,True,False,Day,Day,Day,Day


### STEP 0.2 Parse datetime

In [2]:
datetime_cols = ['Start_Time', 'End_Time', 'Weather_Timestamp']

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

df[datetime_cols].dtypes


Start_Time           datetime64[us]
End_Time             datetime64[us]
Weather_Timestamp    datetime64[us]
dtype: object

### STEP 1: Recreate Time Features 

* โดยจะไม่ดึงจาก EDA — สร้างใหม่ให้ deterministic

In [3]:
df['Start_Hour'] = df['Start_Time'].dt.hour
df['Start_Weekday'] = df['Start_Time'].dt.weekday
df['Start_Month'] = df['Start_Time'].dt.month

df['End_Hour'] = df['End_Time'].dt.hour
df['End_Weekday'] = df['End_Time'].dt.weekday

df['Is_Weekend'] = df['Start_Weekday'].isin([5, 6]).astype(int)

df['Accident_Duration_Min'] = (
    df['End_Time'] - df['Start_Time']
).dt.total_seconds() / 60


### STEP 2: Define Column Groups
* นี่คือ “สัญญา” ระหว่างคุณกับโมเดล

In [4]:
geo_cols = ['Start_Lat', 'Start_Lng', 'End_Lat', 'End_Lng']

weather_numeric = [
    'Temperature(F)', 'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)',
    'Precipitation(in)', 'Wind_Chill(F)'
]

time_cols = [
    'Start_Hour', 'Start_Weekday', 'Start_Month',
    'End_Hour', 'End_Weekday', 'Accident_Duration_Min'
]

categorical_cols = [
    'Severity', 'State', 'Weather_Condition',
    'Sunrise_Sunset'
]

boolean_cols = [
    'Amenity', 'Bump', 'Crossing', 'Give_Way',
    'Junction', 'No_Exit', 'Railway', 'Roundabout',
    'Station', 'Stop', 'Traffic_Calming',
    'Traffic_Signal', 'Turning_Loop'
]


### STEP 3: Handle Missing — แบบที่ “วิเคราะห์มาแล้ว”

3.1 End Location & Duration (MNAR)
* ไม่ impute
* สร้าง flag

df['Has_End_Info'] = df['End_Lat'].notna().astype(int)


3.2 Weather — Missing by Design

Precipitation

In [5]:
df['Has_Precipitation'] = df['Precipitation(in)'].notna().astype(int)
df['Precipitation(in)'] = df['Precipitation(in)'].fillna(0)


Wind Chill

In [6]:
df['Wind_Speed(mph)'] = (
    df.groupby('State')['Wind_Speed(mph)']
      .transform(lambda x: x.fillna(x.median()))
)


3.4 Categorical Missing

In [7]:
df['Weather_Condition'] = df['Weather_Condition'].fillna('Unknown')
df['Sunrise_Sunset'] = df['Sunrise_Sunset'].fillna('Unknown')


### STEP 4: Validate

In [8]:
missing_summary = (
    df.isna()
      .mean()
      .sort_values(ascending=False)
)

missing_summary[missing_summary > 0]


End_Lng                 0.4403
End_Lat                 0.4403
Wind_Chill(F)           0.2587
End_Time                0.0962
Start_Time              0.0962
End_Weekday             0.0962
Start_Weekday           0.0962
Start_Hour              0.0962
End_Hour                0.0962
Start_Month             0.0962
Accident_Duration_Min   0.0962
Visibility(mi)          0.0229
Wind_Direction          0.0227
Humidity(%)             0.0225
Temperature(F)          0.0212
Pressure(in)            0.0182
Weather_Timestamp       0.0156
Astronomical_Twilight   0.0030
Nautical_Twilight       0.0030
Civil_Twilight          0.0030
Airport_Code            0.0029
Street                  0.0014
Timezone                0.0010
Zipcode                 0.0002
City                    0.0000
Description             0.0000
dtype: float64

### STEP 5: Save to processed/

In [9]:
OUTPUT_PATH = "../data/processed/accidents_clean.parquet"

df.to_parquet(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)


Saved: ../data/processed/accidents_clean.parquet


# เริ่มจริง 

### STEP 1: Fix Datetime + Derived time features

In [10]:
datetime_cols = [
    'Start_Time', 'End_Time', 'Weather_Timestamp'
]

for col in datetime_cols:
    df[col] = pd.to_datetime(df[col], errors='coerce')

# Derived time features
df['Start_Hour'] = df['Start_Time'].dt.hour
df['Start_Weekday'] = df['Start_Time'].dt.weekday
df['Start_Month'] = df['Start_Time'].dt.month

df['End_Hour'] = df['End_Time'].dt.hour
df['End_Weekday'] = df['End_Time'].dt.weekday

df['Is_Weekend'] = df['Start_Weekday'].isin([5,6])

df['Accident_Duration_Min'] = (
    df['End_Time'] - df['Start_Time']
).dt.total_seconds() / 60


### STEP 2: Clip invalid weather values

In [11]:
# Temperature & Wind Chill
df['Temperature(F)'] = df['Temperature(F)'].clip(-50, 130)
df['Wind_Chill(F)'] = df['Wind_Chill(F)'].clip(-50, 130)

# Pressure
df['Pressure(in)'] = df['Pressure(in)'].clip(26, 32)


### STEP 3: Clip Distance(mi) (long-tail)

In [12]:
upper_dist = df['Distance(mi)'].quantile(0.999)
upper_dist


np.float64(20.398606999999846)

In [13]:
df['Distance(mi)'] = df['Distance(mi)'].clip(upper=upper_dist)


### STEP 4: Missing Value Strategy (ล็อกแล้ว)

In [14]:
# Numeric weather → median
weather_num = [
    'Temperature(F)', 'Wind_Chill(F)',
    'Humidity(%)', 'Pressure(in)',
    'Visibility(mi)', 'Wind_Speed(mph)',
    'Precipitation(in)'
]

for col in weather_num:
    df[col] = df[col].fillna(df[col].median())

# Boolean → False
bool_cols = df.select_dtypes(include='bool').columns
df[bool_cols] = df[bool_cols].fillna(False)

# Categorical → 'Unknown'
cat_cols = df.select_dtypes(include='object').columns
for col in cat_cols:
    df[col] = df[col].fillna('Unknown')


C:\Users\Lenovo\AppData\Local\Temp\ipykernel_15796\3701976284.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = df.select_dtypes(include='object').columns


### STEP 5: Save cleaned data

In [15]:
OUTPUT_PATH = "../data/processed/accidents_clean.parquet"
df.to_parquet(OUTPUT_PATH, index=False)


In [16]:
df.shape
df.isna().mean().sort_values(ascending=False).head(10)


End_Lng                 0.4403
End_Lat                 0.4403
End_Time                0.0962
Start_Time              0.0962
Accident_Duration_Min   0.0962
End_Weekday             0.0962
Start_Weekday           0.0962
Start_Hour              0.0962
End_Hour                0.0962
Start_Month             0.0962
dtype: float64

### ลองอ่านไฟล์ที่ Clean แล้ว

In [19]:
import pandas as pd

# ลองอ่านไฟล์ที่ Clean แล้ว
df = pd.read_parquet('../data/processed/accidents_clean.parquet')
display(df.head()) # ดู 5 แถวแรก

,ID,Source,Severity,Start_Time,End_Time,Start_Lat,Start_Lng,End_Lat,End_Lng,Distance(mi),Description,Street,City,County,State,Zipcode,Country,Timezone,Airport_Code,Weather_Timestamp,Temperature(F),Wind_Chill(F),Humidity(%),Pressure(in),Visibility(mi),Wind_Direction,Wind_Speed(mph),Precipitation(in),Weather_Condition,Amenity,Bump,Crossing,Give_Way,Junction,No_Exit,Railway,Roundabout,Station,Stop,Traffic_Calming,Traffic_Signal,Turning_Loop,Sunrise_Sunset,Civil_Twilight,Nautical_Twilight,Astronomical_Twilight,Start_Hour,Start_Weekday,Start_Month,End_Hour,End_Weekday,Is_Weekend,Accident_Duration_Min,Has_Precipitation
0,A-1,Source2,3,2016-02-08 05:46:00,2016-02-08 11:00:00,39.8651,-84.0587,NaN,NaN,0.0100,Right lane block...,I-70 E,Dayton,Montgomery,OH,45424,US,US/Eastern,KFFO,2016-02-08 05:58:00,36.9000,62.0000,91.0000,29.6800,10.0000,Calm,8.1000,0.0200,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Night,5.0000,0.0000,2.0000,11.0000,0.0000,False,314.0000,1
1,A-2,Source2,2,2016-02-08 06:07:59,2016-02-08 06:37:59,39.9281,-82.8312,NaN,NaN,0.0100,Accident on Bric...,Brice Rd,Reynoldsburg,Franklin,OH,43068-3402,US,US/Eastern,KCMH,2016-02-08 05:51:00,37.9000,62.0000,100.0000,29.6500,10.0000,Calm,8.1000,0.0000,Light Rain,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Night,Night,Day,6.0000,0.0000,2.0000,6.0000,0.0000,False,30.0000,1
2,A-3,Source2,2,2016-02-08 06:49:27,2016-02-08 07:19:27,39.0631,-84.0326,NaN,NaN,0.0100,Accident on OH-3...,State Route 32,Williamsburg,Clermont,OH,45176,US,US/Eastern,KI69,2016-02-08 06:56:00,36.0000,33.3000,100.0000,29.6700,10.0000,SW,3.5000,0.0000,Overcast,False,False,False,False,False,False,False,False,False,False,False,True,False,Night,Night,Day,Day,6.0000,0.0000,2.0000,7.0000,0.0000,False,30.0000,0
3,A-4,Source2,3,2016-02-08 07:23:34,2016-02-08 07:53:34,39.7478,-84.2056,NaN,NaN,0.0100,Accident on I-75...,I-75 S,Dayton,Montgomery,OH,45417,US,US/Eastern,KDAY,2016-02-08 07:38:00,35.1000,31.0000,96.0000,29.6400,9.0000,SW,4.6000,0.0000,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,False,False,Night,Day,Day,Day,7.0000,0.0000,2.0000,7.0000,0.0000,False,30.0000,0
4,A-5,Source2,2,2016-02-08 07:39:07,2016-02-08 08:09:07,39.6278,-84.1884,NaN,NaN,0.0100,Accident on McEw...,Miamisburg Cente...,Dayton,Montgomery,OH,45459,US,US/Eastern,KMGY,2016-02-08 07:53:00,36.0000,33.3000,89.0000,29.6500,6.0000,SW,3.5000,0.0000,Mostly Cloudy,False,False,False,False,False,False,False,False,False,False,False,True,False,Day,Day,Day,Day,7.0000,0.0000,2.0000,8.0000,0.0000,False,30.0000,0
